### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql = """
SELECT *
FROM "Order" where order_status IN ('Completed', 'Shipped');
"""

df = pd.read_sql(sql, engine)

# 查看数据
df

### Debug: check out df columns name

In [ ]:
df.columns

### Data aggregation and Analysis
### Aggregate each customer `total_order` and `total_spend` and `preferred_store_id`

In [ ]:
import numpy as np
agg = df.groupby("customer_id").agg(
    total_orders=("order_id", "count"),
    total_spend=("total_price_after_tax", "sum"),
    last_purchase_date=("approval_date", "max"),
    preferred_store_id=("store_id", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
).reset_index()

In [ ]:
agg

### fillin `total_order` and `total_spend` and `preferred_store_id`

In [ ]:
with engine.begin() as conn:
    agg.to_sql("tmp_customer_agg", conn, if_exists="replace", index=False)
    conn.execute(text("""
        UPDATE "CustomerInfo" c
        SET
            total_orders = t.total_orders,
            total_spend = t.total_spend,
            last_purchase_date = t.last_purchase_date,
            preferred_store_id = t.preferred_store_id
        FROM tmp_customer_agg t
        WHERE c.customer_id = t.customer_id;
    """))
    conn.execute(text('DROP TABLE IF EXISTS tmp_customer_agg;'))

In [ ]:
customer_df = pd.read_sql('SELECT * FROM "CustomerInfo";', engine)
customer_df

### Aggregate each customer `channel_source`

In [ ]:
import numpy as np
import pandas as pd

# ====================== 设置随机种子，保证结果可重复 ======================
np.random.seed(42)

# 已读取订单数据 df（用来筛选有订单的客户）
eligible_ids = df['customer_id'].unique()

# 读取 customerinfo
customer_df_1 = pd.read_sql('SELECT * FROM "CustomerInfo";', engine)

# 只对有订单的客户进行赋值（避免给无订单客户乱赋值）
mask = customer_df_1['customer_id'].isin(eligible_ids)

# ====================== 核心赋值逻辑 ======================
def get_channel_prob(age):
    """根据年龄返回 [Offline概率, Online概率]"""
    if age <= 30:
        return [0.62, 0.38]   # 年轻人线上比例较高
    elif age <= 45:
        return [0.75, 0.25]
    else:
        return [0.85, 0.15]   # 年长者以线下为主

# 计算每个客户的概率
probs = customer_df_1.loc[mask, 'age'].apply(get_channel_prob)

# 生成 channel_source
customer_df_1.loc[mask, 'channel_source'] = [
    np.random.choice(['Offline', 'Online'], p=p) 
    for p in probs
]

# ====================== 结果检查 ======================
print(customer_df_1['channel_source'].value_counts(normalize=True))
print("\n各年龄段渠道分布：")

age_group = (customer_df_1['age_group'] if 'age_group' in customer_df_1.columns else pd.cut(customer_df_1['age'], bins=[0,30,45,100], labels=['<=30', '31-45', '>45']))
print(customer_df_1.groupby(age_group, observed=False)['channel_source'].value_counts(normalize=True))

customer_df_1

### fillin each customer `channel_source` data

In [ ]:
# 只保留有效的 channel_source
mask = customer_df_1['channel_source'].notna() & (customer_df_1['channel_source'] != '')
updated = customer_df_1.loc[mask, ['customer_id', 'channel_source']].copy()

with engine.begin() as conn:          # 使用事务，保证要么全成功要么全失败
    # 创建临时表并写入数据
    updated.to_sql('tmp_channel_source', conn, if_exists='replace', index=False)
    
    # 执行更新
    conn.execute(text("""
        UPDATE "CustomerInfo" c
        SET channel_source = t.channel_source
        FROM tmp_channel_source t
        WHERE c.customer_id = t.customer_id;
    """))
    
    # 删除临时表
    conn.execute(text('DROP TABLE IF EXISTS tmp_channel_source;'))

print(f"✅ 已成功更新 {len(updated)} 条记录的 channel_source 到 CustomerInfo 表")

In [ ]:
# 关闭数据库连接
engine.dispose()